# Corrida nueva: descarga, triaje y 4 prompts

Corrida de prueba **`a2_test`** (sector A2 aproximado, ver `configs/a2_test.yaml`). Todo queda en `MyDrive/sepia-inaturalist/data/a2_test/`, separado del piloto.

- Los registros que ya están en `pilot_a1` **no se vuelven a bajar** (`skip_ids_from`); quedan listados en `logs/skipped_existing.jsonl`.
- Todo es reanudable: si una celda se corta (por ejemplo, por el límite diario de Gemini), se vuelve a correr la misma celda y sigue donde quedó.
- Este notebook **no muestra los resultados de los modelos**: así no se rompe una anotación a ciegas posterior.

Para otra corrida: copiar `configs/a2_test.yaml`, cambiar `base_dir`, `bbox` y `skip_ids_from`, y cambiar `CONFIG` abajo.

In [ ]:
# --- Preparacion ---
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys, importlib
if not os.path.exists('/content/sepia-inaturalist'):     # sesion nueva: clonar el repo
    token = userdata.get('GITHUB_TOKEN')
    !git clone -q https://{token}@github.com/hugozeballos/sepia-inaturalist.git /content/sepia-inaturalist
%cd /content/sepia-inaturalist
!git pull -q
!git log -1 --oneline
!pip install -q -e .

os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')   # nunca en el codigo

sys.path.insert(0, '/content/sepia-inaturalist/src')
for m in [m for m in sys.modules if m == 'sepia' or m.startswith('sepia.')]:
    del sys.modules[m]
sys.path_importer_cache.pop('/content/sepia-inaturalist/src', None)
importlib.invalidate_caches()
import sepia

CONFIG = 'configs/a2_test.yaml'
VERSIONS = ['photo_traits_v14', 'photo_traits_v13', 'photo_traits_v12', 'photo_traits_v8']
cfg = sepia.load_config(CONFIG)
print(cfg.base_dir, cfg.bbox, 'salta:', cfg.skip_ids_from)

## 1. ¿Cuántos registros?

Total de la consulta (incluye los que ya están en `pilot_a1`; esos se descuentan en el paso 2).

In [ ]:
!python scripts/download.py --config {CONFIG} --count-only

## 2. Descarga: metadatos y tablas

Baja la copia cruda de la API y arma las tablas **sin** los registros que ya están en otras corridas. Muestra cuántos se saltan.

In [ ]:
!python scripts/download.py --config {CONFIG} --steps metadata,tables

## 3. Descarga: fotos, versiones reducidas y chequeo

In [ ]:
!python scripts/download.py --config {CONFIG} --steps photos,resize,check

## 4. Triaje (`triage_v2`): estado y número de huesos por registro

In [ ]:
!python scripts/triage.py --config {CONFIG}

## 5. Los 4 prompts

Una consulta por registro, 8 por minuto. Si se corta, volver a correr: las fotos hechas se saltan.

In [ ]:
for v in VERSIONS:
    print(f"\n===== {v} =====")
    !python scripts/photo_traits.py --config {CONFIG} --prompt-version {v}

## 6. Resumen (sin mostrar resultados de los modelos)

In [ ]:
obs, pho = sepia.load_tables(cfg)
p = sepia.paths(cfg)
skipped = sepia.read_jsonl(p['logs'] / 'skipped_existing.jsonl')
print(f"Registros: {len(obs)} · fotos: {len(pho)} · saltados (ya en otra corrida): {len(skipped)}")

t = cfg.triage
tri = sepia.read_jsonl(sepia.triage.output_path(cfg, t['provider'], t['model'], t['prompt_version']))
if len(tri):
    ok = tri[tri['error'].isna()]['observation_id'].nunique() if 'error' in tri else tri['observation_id'].nunique()
    print(f"Triaje: {ok}/{len(obs)} registros sin error")

for v in VERSIONS:
    tc = {**cfg.photo_traits, 'prompt_version': v}
    raw = sepia.io.read_model_outputs(cfg, 'photo_traits', tc['provider'], tc['model'], v)
    if raw.empty:
        print(f"{v}: sin resultados todavia")
        continue
    last = raw.sort_values('saved_at').drop_duplicates('photo_id', keep='last')
    n_err = int(last['error'].notna().sum())
    print(f"{v}: {int(last['error'].isna().sum())}/{len(pho)} fotos sin error · con error: {n_err}"
          + (" (volver a correr el paso 5)" if n_err or len(last) < len(pho) else ""))